# Multilayer Perceptron with Sequential Wrapper

## Libraries

In [60]:
import time
import numpy as np
from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torch

In [61]:
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True

## Settings

In [62]:
# Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Hyperparameters
random_seed = 1
learning_rate = 0.1
num_epochs = 10
batch_size = 64

# Architecture
num_features = 784
num_hidden_1 = 128
num_hidden_2 = 256
num_classes = 10

## MNIST Dataset

In [63]:
train_dataset = datasets.MNIST(root='data', 
                               train=True, 
                               transform=transforms.ToTensor(),
                               download=True)

test_dataset = datasets.MNIST(root='data', 
                              train=False, 
                              transform=transforms.ToTensor())

train_loader = DataLoader(dataset=train_dataset, 
                          batch_size=batch_size, 
                          shuffle=True)

test_loader = DataLoader(dataset=test_dataset, 
                         batch_size=batch_size, 
                         shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 12.7MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 337kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 3.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.34MB/s]


In [64]:
for images, labels in train_loader:  
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])


## Model

In [65]:
class MultilayerPerceptron(torch.nn.Module):

    def __init__(self, num_features, num_classes):
        super(MultilayerPerceptron, self).__init__()
        
        self.net = torch.nn.Sequential(
            torch.nn.Linear(num_features, num_hidden_1),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(num_hidden_1, num_hidden_2),
            torch.nn.ReLU(inplace=True),
            torch.nn.Linear(num_hidden_2, num_classes)
        )
        
    def forward(self, x):
        logits = self.net(x)
        probas = F.log_softmax(logits, dim=1)
        return logits, probas

torch.manual_seed(random_seed)
model = MultilayerPerceptron(num_features=num_features,
                             num_classes=num_classes)

model = model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)  

In [66]:
def compute_accuracy(net, data_loader):
    net.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.view(-1, 28*28).to(device)
            targets = targets.to(device)
            logits, probas = net(features)
            _, predicted_labels = torch.max(probas, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
        return correct_pred.float()/num_examples * 100
    

start_time = time.time()
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, targets) in enumerate(train_loader):
        
        features = features.view(-1, 28*28).to(device)
        targets = targets.to(device)
            
        ### FORWARD AND BACK PROP
        logits, probas = model(features)
        cost = F.cross_entropy(logits, targets)
        optimizer.zero_grad()
        
        cost.backward()
        
        ### UPDATE MODEL PARAMETERS
        optimizer.step()
        
        ### LOGGING
        if not batch_idx % 50:
            print ('Epoch: %03d/%03d | Batch %03d/%03d | Cost: %.4f' 
                   %(epoch+1, num_epochs, batch_idx, 
                     len(train_loader), cost))

    with torch.set_grad_enabled(False):
        print('Epoch: %03d/%03d training accuracy: %.2f%%' % (
              epoch+1, num_epochs, 
              compute_accuracy(model, train_loader)))
        
    print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    
print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))

Epoch: 001/010 | Batch 000/938 | Cost: 2.3005
Epoch: 001/010 | Batch 050/938 | Cost: 1.7801
Epoch: 001/010 | Batch 100/938 | Cost: 0.7709
Epoch: 001/010 | Batch 150/938 | Cost: 0.5032
Epoch: 001/010 | Batch 200/938 | Cost: 0.5423
Epoch: 001/010 | Batch 250/938 | Cost: 0.2938
Epoch: 001/010 | Batch 300/938 | Cost: 0.3381
Epoch: 001/010 | Batch 350/938 | Cost: 0.4302
Epoch: 001/010 | Batch 400/938 | Cost: 0.1890
Epoch: 001/010 | Batch 450/938 | Cost: 0.2354
Epoch: 001/010 | Batch 500/938 | Cost: 0.2359
Epoch: 001/010 | Batch 550/938 | Cost: 0.2611
Epoch: 001/010 | Batch 600/938 | Cost: 0.3027
Epoch: 001/010 | Batch 650/938 | Cost: 0.3592
Epoch: 001/010 | Batch 700/938 | Cost: 0.4079
Epoch: 001/010 | Batch 750/938 | Cost: 0.2695
Epoch: 001/010 | Batch 800/938 | Cost: 0.2783
Epoch: 001/010 | Batch 850/938 | Cost: 0.1079
Epoch: 001/010 | Batch 900/938 | Cost: 0.3135
Epoch: 001/010 training accuracy: 92.80%
Time elapsed: 0.21 min
Epoch: 002/010 | Batch 000/938 | Cost: 0.1650
Epoch: 002/010 |

In [67]:
print('Test accuracy: %.2f%%' % (compute_accuracy(model, test_loader)))

Test accuracy: 97.83%


## Accessing Intermediate Results via Hooks

In [68]:
model.net

Sequential(
  (0): Linear(in_features=784, out_features=128, bias=True)
  (1): ReLU(inplace=True)
  (2): Linear(in_features=128, out_features=256, bias=True)
  (3): ReLU(inplace=True)
  (4): Linear(in_features=256, out_features=10, bias=True)
)

In [69]:
outputs = []
def hook(module, input, output):
    outputs.append(output)

model.net[2].register_forward_hook(hook)

In [70]:
_ = model(features)

print(outputs)

[tensor([[0.0000, 0.0000, 0.1963,  ..., 0.0000, 0.0000, 0.9336],
        [0.4378, 0.0732, 0.6228,  ..., 0.0285, 0.0000, 0.3828],
        [0.9206, 0.3051, 0.9247,  ..., 0.0000, 0.0000, 0.3247],
        ...,
        [1.7131, 0.1801, 0.0000,  ..., 1.6257, 0.2395, 0.8900],
        [1.2648, 0.5694, 0.1884,  ..., 0.5951, 0.0000, 0.0000],
        [0.3370, 0.0721, 0.0000,  ..., 0.8154, 0.0000, 1.7123]],
       device='cuda:0', grad_fn=<ReluBackward0>)]
